# Lenormand Task1 + B4-E2 — Qwen3.8 Full64 Outer Fold 1/2 Confirmation

本 notebook 做最后一次不调参确认：

```text
Fold-specific Qwen3.8 Full64 Task-1 adapter
  └─ adapter for Fold k is trained on folds != k
        ↓
fold-safe ModernBERT proposer + evidence lexicon
        ↓
candidate margins + document/conditioned/blended risk
        ↓
frozen Fold-0 B4-E2 meta-calibrator
        ↓
official one-to-one Evidence F1 + Risk Weighted-F1
```

绝对禁止在看 Fold1/2 后修改 `.40 / gap40 / k0232 / Logistic C=.1 / features`。

为了避免 Colab 单次运行时间过长，默认每个 session 只跑一折：先保持 `FOLDS_TO_RUN=(1,)`；Fold1 完成后改为 `(2,)`。训练 checkpoint、proposer 和三个评分阶段都保存在 Drive，可以续跑。A100 80GB 每折通常约 4–8 小时，实际取决于 Colab 吞吐。

In [ ]:
#@title 0A. 安装基础依赖
%%capture
!pip install -q -U \
  "transformers>=5.8.0" \
  "accelerate>=1.6.0" \
  "peft>=0.17.0" \
  "bitsandbytes>=0.46.0" \
  "sentence-transformers>=3.4.0" \
  "sentencepiece>=0.2.0" \
  "openpyxl>=3.1.0" \
  "scikit-learn==1.7.2" \
  "scipy>=1.13.0" \
  "kernels"


In [ ]:
#@title 0B. 安装 Qwen3.8 混合架构 kernels
!pip install -U "flash-linear-attention[cuda]"
!pip install -U causal-conv1d --no-build-isolation
print('安装后执行 Runtime → Restart session；重启后从第 1 格继续，不要重跑 0A/0B。')

> 必须重启一次 runtime，让 Qwen3.8 重新探测 Flash Linear Attention 与 causal-conv1d。重启后从第 1 格继续。

In [ ]:
#@title 1. Drive、运行折与路径
from google.colab import drive, files
drive.mount('/content/drive')

from pathlib import Path
import dataclasses, gc, importlib, json, shutil, subprocess, sys, time, zipfile

ROOT = Path('/content/drive/MyDrive/IEEE_BigData2026')
TRAIN_PATH = ROOT / 'train.xlsx'
FOLD_REFERENCE = ROOT / 'results' / 'B4P_AVC_FAST3' / 'B4P_CORE_OOF.npz'
FOLD0_TASK1_ROOT = ROOT / 'results' / 'B4_TASK1_Q38_FULL64_FOLD0'
CONFIRM_ROOT = ROOT / 'results' / 'B4_TASK1_Q38_FULL64_OUTER_CONFIRM'
FOLD_ROOT = CONFIRM_ROOT / 'Q38_FULL64'
PROPOSER_ROOT = CONFIRM_ROOT / 'MODERNBERT_PROPOSER'
B4E2_ROOT = ROOT / 'results' / 'B4E2_CANDIDATE_META_FOLD0'
B4E2_CONFIRM_ROOT = CONFIRM_ROOT / 'B4E2_OUTER_CONFIRMATION'

# 推荐一折一个 Colab session。Fold1 完成后只改成 (2,)。
FOLDS_TO_RUN = (1,)
RUN_TRAIN = True
RUN_SCORE = True
OVERWRITE_ADAPTERS = False
FACTOR_MACRO_F1 = 0.691954
RANK8_REFERENCE = 0.7615

for fold in FOLDS_TO_RUN:
    assert fold in (1, 2), '确认 notebook 只允许 Fold1/2'
CONFIRM_ROOT.mkdir(parents=True, exist_ok=True)
FOLD_ROOT.mkdir(parents=True, exist_ok=True)

MODULES = {
    'b1_experiments.py': None,
    'b1_innovation_experiments.py': None,
    'b4p_anchor_verifier.py': 'B4P_RUNTIME_REVISION = \"2026-08-21.qwen38-full64-kernels-v4\"',
    'qwen38_dual_task_experiments.py': 'Q38_RUNTIME_REVISION = \"2026-08-24.official-evidence-scorer-v3\"',
    'b4_task1_q38.py': 'TASK1_RUNTIME_REVISION = \"2026-08-24.q38-full64-official-evidence-v4\"',
    'b4e_evidence_set.py': 'B4E_RUNTIME_REVISION = \"2026-08-24.official-one-to-one-event-set-v1\"',
    'b4e_candidate_meta.py': 'def confirm_frozen_meta_decoder(',
}
stale = []
for name, marker in MODULES.items():
    path = ROOT / name
    if not path.exists() or (marker and marker not in path.read_text(encoding='utf-8')):
        stale.append(name)
if stale:
    print('请一次性上传并覆盖：', stale)
    uploaded = files.upload()
    for name in stale:
        if name not in uploaded:
            raise FileNotFoundError(name)
        shutil.copy2('/content/' + name, ROOT / name)

# 若 Drive 没有 B4-E2 文件，可直接上传上一步的 zip。
meta_model_path = B4E2_ROOT / 'B4E2_FOLD0_META_CALIBRATOR.joblib'
meta_config_path = B4E2_ROOT / 'B4E2_FROZEN_CONFIG.json'
if not meta_model_path.exists() or not meta_config_path.exists():
    print('请上传 B4E2_CANDIDATE_META_FOLD0.zip')
    uploaded = files.upload()
    zip_name = 'B4E2_CANDIDATE_META_FOLD0.zip'
    if zip_name not in uploaded:
        raise FileNotFoundError(zip_name)
    B4E2_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile('/content/' + zip_name) as archive:
        archive.extractall(B4E2_ROOT)

for path in [TRAIN_PATH, FOLD_REFERENCE, meta_model_path, meta_config_path]:
    assert path.exists(), path
sys.path.insert(0, str(ROOT))
print(subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    capture_output=True, text=True,
).stdout)
print({'run_folds': FOLDS_TO_RUN, 'artifacts': str(CONFIRM_ROOT)})

In [ ]:
#@title 2. Kernel 硬检查、模块版本与 Fold-0 冻结配置
import numpy as np
import pandas as pd
import torch
import transformers
import sklearn
import b1_experiments as b1
import b1_innovation_experiments as inn
import b4p_anchor_verifier as b4
import qwen38_dual_task_experiments as q38
import b4_task1_q38 as t1
import b4e_evidence_set as b4e
import b4e_candidate_meta as b4e2
importlib.reload(b1); importlib.reload(inn); importlib.reload(b4); importlib.reload(q38)
importlib.reload(t1); importlib.reload(b4e); importlib.reload(b4e2)

assert b4.B4P_RUNTIME_REVISION == '2026-08-21.qwen38-full64-kernels-v4'
assert q38.Q38_RUNTIME_REVISION == '2026-08-24.official-evidence-scorer-v3'
assert t1.TASK1_RUNTIME_REVISION == '2026-08-24.q38-full64-official-evidence-v4'
assert b4e2.B4E2_RUNTIME_REVISION == '2026-08-24.candidate-meta-event-decoder-v2'
kernel_status = b4.qwen35_kernel_status()
print({'transformers': transformers.__version__, 'torch': torch.__version__, 'sklearn': sklearn.__version__, 'kernel': kernel_status})
assert sklearn.__version__ == '1.7.2', '冻结 meta model 必须使用 scikit-learn 1.7.2 加载'
assert kernel_status['causal_conv1d']
assert kernel_status['flash_linear_attention']
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
assert gpu_memory_gb >= 70, f'Full64 要求 A100 80GB；当前 {gpu_memory_gb:.1f}GB'
torch.set_float32_matmul_precision('high')

selected_path = FOLD0_TASK1_ROOT / 'TASK1_CONFIG_SELECTED.json'
spec_path = FOLD0_TASK1_ROOT / 'Q38_FULL64' / 'fold_0' / 'frozen_task1_spec.json'
if selected_path.exists():
    frozen_task1 = json.loads(selected_path.read_text(encoding='utf-8'))
elif spec_path.exists():
    frozen_task1 = json.loads(spec_path.read_text(encoding='utf-8'))['config']
else:
    raise FileNotFoundError('缺少 Fold-0 TASK1_CONFIG_SELECTED.json/frozen_task1_spec.json')
for field in ('candidate_caps_for_audit', 'lora_target_leaves'):
    if field in frozen_task1:
        frozen_task1[field] = tuple(frozen_task1[field])
BASE_CFG = t1.Task1Full64Config(**frozen_task1)
assert BASE_CFG.model_name == 'Qwen/Qwen3.8-27B'
assert BASE_CFG.lora_last_n_layers is None
assert BASE_CFG.attention_implementation == 'flash_attention_2'
assert BASE_CFG.conditioned_risk_blend_weight == 0.65

frozen_meta = json.loads(meta_config_path.read_text(encoding='utf-8'))
assert frozen_meta == dataclasses.asdict(b4e2.CandidateMetaConfig())
print('Frozen Task1 candidate cap:', BASE_CFG.validation_candidates_per_post)
print('Frozen B4-E2:', frozen_meta)

In [ ]:
#@title 3. 数据、相同 grouped folds 与证据审计
bundle = b1.load_training_data(ROOT, TRAIN_PATH)
reference = np.load(FOLD_REFERENCE, allow_pickle=True)
assert reference['row_ids'].astype(str).tolist() == bundle.row_ids.astype(str).tolist()
folds = reference['folds'].astype(int)
annotations, annotation_report = inn.prepare_evidence_annotations(bundle)
fold_hash = b4.stable_hash({'row_ids': bundle.row_ids.tolist(), 'folds': folds.tolist()})
print('fold sizes:', np.bincount(folds).tolist(), 'hash:', fold_hash)
print(b4.validate_folds(bundle, folds))
print('evidence annotations:', annotation_report)
for fold in FOLDS_TO_RUN:
    print(f'Fold {fold}: train={(folds != fold).sum()}, valid={(folds == fold).sum()}')

In [ ]:
#@title 4. Fold-safe ModernBERT proposer（完成后自动复用）
proposers = {}
for fold in FOLDS_TO_RUN:
    print(f'\n========== PREPARE PROPOSER FOLD {fold} ==========')
    proposers[fold] = t1.prepare_modernbert_proposer_fold(
        bundle, annotations, folds, fold, PROPOSER_ROOT, epochs=2,
    )
    print(json.dumps(proposers[fold].metrics, indent=2, default=str))

In [ ]:
#@title 5. 训练/恢复 Full64 Task-1 adapter（每折约 3–5 小时）
adapters = {}
lexicons = {}
if RUN_TRAIN:
    for fold in FOLDS_TO_RUN:
        cfg = dataclasses.replace(BASE_CFG, fold=fold)
        print(f'\n========== TRAIN / RESUME TASK1 FULL64 FOLD {fold} ==========')
        started = time.perf_counter()
        adapter, lexicon, manifest = t1.train_fold(
            bundle, annotations, folds, cfg, FOLD_ROOT,
            overwrite=OVERWRITE_ADAPTERS,
        )
        adapters[fold] = adapter
        lexicons[fold] = lexicon
        print({
            'fold': fold, 'adapter': str(adapter), 'pairs': len(manifest),
            'session_hours': (time.perf_counter() - started) / 3600,
        })
else:
    print('RUN_TRAIN=False')

In [ ]:
#@title 6. Document → Evidence → Conditioned Risk 评分（分块续跑）
fold_metrics = {}
if RUN_SCORE:
    for fold in FOLDS_TO_RUN:
        cfg = dataclasses.replace(BASE_CFG, fold=fold)
        fold_dir = FOLD_ROOT / f'fold_{fold}'
        adapter = fold_dir / 'adapter' / 'adapter_final'
        lexicon_path = fold_dir / 'evidence_lexicon.json'
        assert (adapter / 'adapter_config.json').exists(), adapter
        lexicon = t1.EvidenceLexicon.from_json(json.loads(lexicon_path.read_text()))
        eval_dir = fold_dir / 'EVALUATION'
        metrics_path = eval_dir / 'q38_task1_fold_metrics.json'
        print(f'\n========== SCORE / RESUME TASK1 FOLD {fold} ==========')
        if metrics_path.exists():
            fold_metrics[fold] = json.loads(metrics_path.read_text())
            print('[resume complete]', metrics_path)
        else:
            fold_metrics[fold] = t1.evaluate_fold(
                bundle, folds, adapter, lexicon, cfg, eval_dir,
                token_proposals=proposers[fold],
            )
        print(json.dumps(fold_metrics[fold], indent=2, default=str))
else:
    print('RUN_SCORE=False')

In [ ]:
#@title 7. 检查 Fold1/2 是否齐全；缺一折时安全停止
fold_artifacts = {}
for fold in (1, 2):
    eval_dir = FOLD_ROOT / f'fold_{fold}' / 'EVALUATION'
    audit_path = eval_dir / 'evidence_candidate_audit.csv'
    prediction_path = eval_dir / 'validation_predictions.csv'
    metrics_path = eval_dir / 'q38_task1_fold_metrics.json'
    complete = audit_path.exists() and prediction_path.exists() and metrics_path.exists()
    print({'fold': fold, 'complete': complete, 'eval_dir': str(eval_dir)})
    if complete:
        fold_artifacts[fold] = (audit_path, prediction_path)

READY_FOR_CONFIRMATION = set(fold_artifacts) == {1, 2}
if not READY_FOR_CONFIRMATION:
    print('尚未齐全：下一 session 把 FOLDS_TO_RUN 改成缺失折，重跑第 1–7 格。已有结果不会重做。')
else:
    print('Fold1/2 complete: ready for one-shot frozen B4-E2 confirmation.')

In [ ]:
#@title 8. 一次性应用冻结 B4-E2；不搜索任何参数
outer_decision = None
if READY_FOR_CONFIRMATION:
    outer_decision = b4e2.confirm_frozen_meta_decoder(
        fold_artifacts=fold_artifacts,
        bundle=bundle,
        folds=folds,
        model_path=meta_model_path,
        config_path=meta_config_path,
        output_dir=B4E2_CONFIRM_ROOT,
        factor_macro_f1=FACTOR_MACRO_F1,
        rank8_reference=RANK8_REFERENCE,
    )
    print(json.dumps(outer_decision, ensure_ascii=False, indent=2))
else:
    print('SKIP：Fold1/2 未齐全。')

In [ ]:
#@title 9. 最终确认表与决策
if outer_decision is not None:
    display(pd.read_csv(B4E2_CONFIRM_ROOT / 'B4E2_OUTER_FOLD_RESULTS.csv'))
    pooled = outer_decision['pooled']
    display(pd.DataFrame([
        {
            'system': 'INDICATOR_EMPTY_STRONG_BASELINE',
            'risk_weighted_f1': pooled['risk_weighted_f1'],
            'evidence_f1': pooled['strong_baseline_evidence']['f1'],
            'composite_projection': pooled['strong_baseline_composite_projection'],
        },
        {
            'system': 'FROZEN_B4E2_CANDIDATE_META',
            'risk_weighted_f1': pooled['risk_weighted_f1'],
            'evidence_f1': pooled['meta_evidence']['f1'],
            'composite_projection': pooled['meta_composite_projection'],
        },
    ]))
    print({
        'decoder_confirmed': outer_decision['decoder_confirmed'],
        'top8_projection_passed': outer_decision['top8_projection_passed'],
        'accepted_for_final_test': outer_decision['accepted_for_final_test'],
        'next_step': outer_decision['recommended_next_step'],
    })
else:
    print('等待另一折。')

In [ ]:
#@title 10. 打包轻量确认报告（不含 adapters/checkpoints/大 candidate audits）
if outer_decision is not None:
    report_root = CONFIRM_ROOT / 'LIGHT_CONFIRMATION_REPORT'
    report_root.mkdir(parents=True, exist_ok=True)
    for source in [
        B4E2_CONFIRM_ROOT / 'B4E2_OUTER_CONFIRMATION_DECISION.json',
        B4E2_CONFIRM_ROOT / 'B4E2_OUTER_FOLD_RESULTS.csv',
        B4E2_CONFIRM_ROOT / 'B4E2_OUTER_PREDICTIONS.csv',
    ]:
        if source.exists():
            shutil.copy2(source, report_root / source.name)
    for fold in (1, 2):
        eval_dir = FOLD_ROOT / f'fold_{fold}' / 'EVALUATION'
        for name in ('q38_task1_fold_metrics.json', 'validation_predictions.csv'):
            source = eval_dir / name
            if source.exists():
                shutil.copy2(source, report_root / f'fold_{fold}_{name}')
    archive = shutil.make_archive('/content/B4E2_OUTER12_CONFIRMATION', 'zip', report_root)
    print('Saved:', archive)
    # files.download(archive)
else:
    print('Fold1/2 complete 后再打包。')

## 断线恢复

- 不要删除 `B4_TASK1_Q38_FULL64_OUTER_CONFIRM`。
- 新 runtime 安装 kernels 并重启后，从第 1 格继续。
- 第 4 格会复用完成的 proposer。
- 第 5 格会从最新 `checkpoint-*` 继续；存在 `adapter_final` 时直接跳过训练。
- 第 6 格的 document/evidence/conditioned score chunks 都会续跑。
- Fold1 完成后，只把 `FOLDS_TO_RUN=(1,)` 改成 `(2,)`。其他参数一律不改。

## 严谨性边界

每条 Fold1/2 预测都来自没有训练过该行的 base adapter，因此是行级 OOF。由于 Fold0 开发候选的 base adapter 曾在 Fold1/2 上训练，本确认并非完全 nested external test；论文中应称 cross-fitted confirmation。真正外部证据只能来自比赛 test/leaderboard。